In [1]:
import io
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pointbiserialr
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


class PlottingMethods:

    def _to_html(self, fig):
        html = fig.to_html(full_html=False, include_plotlyjs='cdn')
        return {'status': 'success', 'html': html}

    def display_image(self, result):
        if result.get('status') == 'success':
            display(HTML(result['html']))
        else:
            print("Error:", result.get('message'))

    def get_methods_info(self):
        methods = {
            'plot_bar_chart': 'Bar or grouped bar chart',
            'plot_pie_chart': 'Pie or donut chart',
            'plot_histogram': 'Histogram with optional bins',
            'plot_heat_map': 'Pivot heatmap',
            'plot_sankey_diagram': 'Sankey flow diagram',
            'plot_simple_sunburst_graph': 'Sunburst chart',
        }
        return {'response': methods}

    def plot_bar_chart(self, x, y, data, color=None, barmode='group', title=None):
        try:
            if data is None or data.empty:
                return {'status': 'error', 'message': 'No data provided.'}
            fig = px.bar(data, x=x, y=y, color=color, barmode=barmode,
                         title=title or f'{y} by {x}', template='plotly_white')
            return self._to_html(fig)
        except Exception as e:
            return {'status': 'error', 'message': str(e)}

    def plot_pie_chart(self, names, values, data, hole=0.0, title=None):
        try:
            if data is None or data.empty:
                return {'status': 'error', 'message': 'No data provided.'}
            fig = px.pie(data, names=names, values=values, hole=hole,
                         title=title or f'{values} by {names}', template='plotly_white')
            return self._to_html(fig)
        except Exception as e:
            return {'status': 'error', 'message': str(e)}

    def plot_histogram(self, x, data, bins=None, title=None):
        try:
            if data is None or data.empty:
                return {'status': 'error', 'message': 'No data provided.'}
            fig = px.histogram(data, x=x, nbins=len(bins)-1 if bins else None,
                               title=title or f'Distribution of {x}', template='plotly_white')
            return self._to_html(fig)
        except Exception as e:
            return {'status': 'error', 'message': str(e)}

    def plot_heat_map(self, values, index, columns, data, aggregade_method='mean', title=None):
        try:
            if data is None or data.empty:
                return {'status': 'error', 'message': 'No data provided.'}
            pivot = data.pivot_table(values=values, index=index, columns=columns, aggfunc=aggregade_method)
            fig = px.imshow(pivot, text_auto='.2f', template='plotly_white',
                            title=title or f'{aggregade_method} of {values}',
                            color_continuous_scale='RdBu_r')
            return self._to_html(fig)
        except Exception as e:
            return {'status': 'error', 'message': str(e)}

    def plot_sankey_diagram(self, source_column, target_column, values, data, title=None):
        try:
            if data is None or data.empty:
                return {'status': 'error', 'message': 'No data provided.'}
            agg = data.groupby([source_column, target_column])[values].sum().reset_index()
            all_nodes = pd.unique(agg[[source_column, target_column]].values.ravel())
            node_index = {node: i for i, node in enumerate(all_nodes)}
            fig = go.Figure(go.Sankey(
                node=dict(label=list(all_nodes), pad=15, thickness=20),
                link=dict(
                    source=[node_index[s] for s in agg[source_column]],
                    target=[node_index[t] for t in agg[target_column]],
                    value=agg[values].tolist()
                )
            ))
            fig.update_layout(title_text=title or f'{source_column} to {target_column}',
                              template='plotly_white')
            return self._to_html(fig)
        except Exception as e:
            return {'status': 'error', 'message': str(e)}

    def plot_simple_sunburst_graph(self, path, values, data, title=None):
        try:
            if data is None or data.empty:
                return {'status': 'error', 'message': 'No data provided.'}
            fig = px.sunburst(data, path=path, values=values,
                              title=title or ' > '.join(path), template='plotly_white')
            return self._to_html(fig)
        except Exception as e:
            return {'status': 'error', 'message': str(e)}


class DataInspector:

    NULL_STRINGS = ['?', 'n/a', 'na', 'null', 'none', '', ' ', 'nan',
                    'N/A', 'NA', 'NULL', 'None', 'NaN', 'missing']

    def __init__(self):
        self.df = pd.DataFrame()
        self._plotter = PlottingMethods()

    def _check_df(self):
        if self.df.empty:
            print("No data loaded yet.")
            return False
        return True

    def _auto_convert_types(self):
        for col in self.df.columns:
            if self.df[col].dtype == object:
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if converted.notna().any():
                    self.df[col] = converted

    # Data Loading

    def upload_data(self):
        if not IN_COLAB:
            print("Not in Colab. Use: inspector.df = pd.read_csv('file.csv')")
            return
        uploaded = files.upload()
        for filename, content in uploaded.items():
            self.df = pd.read_csv(io.BytesIO(content),
                                  na_values=self.NULL_STRINGS,
                                  keep_default_na=True)
            self._auto_convert_types()
            print(f"Loaded '{filename}' — {self.df.shape[0]} rows x {self.df.shape[1]} cols")
            break

    # Inspection

    def get_summary(self):
        if not self._check_df():
            return
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        print(f"Rows: {self.df.shape[0]}, Columns: {self.df.shape[1]}")
        print(f"Numeric columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
        display(self.df.head(20))
        display(self.df.describe())

    def column_details(self):
        if not self._check_df():
            return
        info = pd.DataFrame({
            'Column': self.df.columns,
            'Dtype': self.df.dtypes.values,
            'Non-Null': self.df.count().values,
            'Null': self.df.isnull().sum().values,
            'Unique': self.df.nunique().values,
        })
        display(info)

    def get_categorical_summary(self):
        if not self._check_df():
            return
        cat_cols = self.df.select_dtypes(exclude='number').columns
        if len(cat_cols) == 0:
            print("No categorical columns found.")
            return
        for col in cat_cols:
            counts = self.df[col].value_counts(dropna=False)
            pct = (counts / len(self.df) * 100).round(2)
            display(pd.DataFrame({'Count': counts, 'Percentage %': pct}))

    def show_missing_data(self):
        if not self._check_df():
            return
        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        if missing.empty:
            print("No missing values found.")
            return
        pct = (missing / len(self.df) * 100).round(2)
        display(pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
                .sort_values('Missing %', ascending=False))

    # Cleaning

    def handle_missing_values(self, strategy='mean', constant=0):
        if not self._check_df():
            return
        num_cols = self.df.select_dtypes(include='number').columns
        if strategy == 'mean':
            self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean())
        elif strategy == 'median':
            self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].median())
        elif strategy == 'mode':
            for col in self.df.columns:
                mode_val = self.df[col].mode()
                if not mode_val.empty:
                    self.df[col] = self.df[col].fillna(mode_val[0])
        elif strategy == 'constant':
            self.df = self.df.fillna(constant)
        else:
            print("Unknown strategy. Use: mean, median, mode, constant")
            return
        print(f"Missing values handled using '{strategy}'.")

    def remove_duplicates(self):
        if not self._check_df():
            return
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"Removed {before - len(self.df)} duplicate(s). Rows remaining: {len(self.df)}")

    def handle_outliers(self, columns=None, find_and_delete=False):
        if not self._check_df():
            return
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        columns = [c for c in columns if c in num_cols] if columns else num_cols
        if not columns:
            print("No valid numeric columns.")
            return
        mask = pd.Series([False] * len(self.df), index=self.df.index)
        for col in columns:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers = (self.df[col] < lower) | (self.df[col] > upper)
            print(f"{col}: {outliers.sum()} outliers (range: {lower:.2f} to {upper:.2f})")
            mask |= outliers
        if find_and_delete:
            self.df = self.df[~mask].reset_index(drop=True)
            print(f"Deleted {mask.sum()} outlier rows. Remaining: {len(self.df)}")
        else:
            print(f"Total outlier rows: {mask.sum()}. Use find_and_delete=True to remove.")

    def delete_rows(self, indices=None):
        if not self._check_df():
            return
        if indices is None:
            indices = input("Enter row indices to delete (comma-separated): ")
        try:
            idx_list = [int(i.strip()) for i in indices.split(',') if i.strip()]
            valid = [i for i in idx_list if i in self.df.index]
            self.df = self.df.drop(index=valid).reset_index(drop=True)
            print(f"Deleted {len(valid)} row(s). Remaining: {len(self.df)}")
        except ValueError:
            print("Please enter valid integers separated by commas.")

    def delete_columns(self, columns=None):
        if not self._check_df():
            return
        if columns is None:
            print("Columns:", list(self.df.columns))
            columns = input("Enter column names to delete (comma-separated): ")
        col_list = [c.strip() for c in columns.split(',') if c.strip()]
        valid = [c for c in col_list if c in self.df.columns]
        self.df = self.df.drop(columns=valid)
        print(f"Deleted: {valid}. Remaining: {list(self.df.columns)}")

    # Feature Engineering

    def extract_normalized_numeric_data(self, method='minmax'):
        if not self._check_df():
            return pd.DataFrame()
        num_cols = self.df.select_dtypes(include='number').columns
        scalers = {'minmax': MinMaxScaler(), 'standard': StandardScaler(), 'robust': RobustScaler()}
        scaler = scalers.get(method)
        if scaler is None:
            print("Unknown method. Use: minmax, standard, robust")
            return pd.DataFrame()
        scaled = scaler.fit_transform(self.df[num_cols])
        print(f"Numeric data scaled using '{method}'.")
        return pd.DataFrame(scaled, columns=num_cols, index=self.df.index)

    def extract_normalized_categorical_data(self, method='onehot'):
        if not self._check_df():
            return pd.DataFrame()
        cat_cols = self.df.select_dtypes(exclude='number').columns
        if len(cat_cols) == 0:
            print("No categorical columns found.")
            return pd.DataFrame()
        if method == 'onehot':
            encoded = pd.get_dummies(self.df[cat_cols], drop_first=False)
            bool_cols = encoded.select_dtypes(include='bool').columns
            encoded[bool_cols] = encoded[bool_cols].astype(int)
        elif method in ('ordinal', 'uniform'):
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            codes = enc.fit_transform(self.df[cat_cols].astype(str))
            encoded = pd.DataFrame(codes, columns=cat_cols, index=self.df.index)
            if method == 'uniform':
                for col in encoded.columns:
                    col_max = encoded[col].max()
                    if col_max > 0:
                        encoded[col] = encoded[col] / col_max
        else:
            print("Unknown method. Use: onehot, ordinal, uniform")
            return pd.DataFrame()
        print(f"Categorical data encoded using '{method}'.")
        return encoded

    def create_normalized_data_df(self, numeric_method='minmax', categorical_method='ordinal'):
        num_df = self.extract_normalized_numeric_data(method=numeric_method)
        cat_df = self.extract_normalized_categorical_data(method=categorical_method)
        parts = [df for df in [num_df, cat_df] if not df.empty]
        if not parts:
            return pd.DataFrame()
        merged = pd.concat(parts, axis=1)
        print(f"Merged DataFrame shape: {merged.shape}")
        return merged

    # Visualisation

    def plot_numerical(self, column_names=None):
        if not self._check_df():
            return
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        column_names = [c for c in column_names if c in num_cols] if column_names else num_cols
        for col in column_names:
            data = self.df[col].dropna()
            fig = make_subplots(rows=1, cols=3,
                                subplot_titles=[f'Violin: {col}', f'Scatter: {col}', f'Histogram: {col}'])
            fig.add_trace(go.Violin(x=data, name=col, box_visible=True,
                                    meanline_visible=True, orientation='h'), row=1, col=1)
            fig.add_trace(go.Scatter(x=list(range(len(data))), y=data.values,
                                     mode='markers', marker=dict(size=3, opacity=0.5)), row=1, col=2)
            fig.add_trace(go.Histogram(x=data), row=1, col=3)
            fig.update_layout(title_text=f'{col}', template='plotly_white',
                              showlegend=False, height=350)
            display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

    def plot_categorical(self, column_names=None):
        if not self._check_df():
            return
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        column_names = [c for c in column_names if c in cat_cols] if column_names else cat_cols
        for col in column_names:
            counts = self.df[col].value_counts(dropna=False)
            pct = (counts / counts.sum() * 100).round(1)
            fig = go.Figure(go.Bar(
                x=counts.index.astype(str), y=counts.values,
                text=[f'{c} ({p}%)' for c, p in zip(counts.values, pct)],
                textposition='auto'
            ))
            fig.update_layout(title=f'{col}', xaxis_title=col,
                              yaxis_title='Count', template='plotly_white')
            display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

    def plot_relationship(self, col_x, col_y):
        if not self._check_df():
            return
        if col_x not in self.df.columns or col_y not in self.df.columns:
            print(f"Column '{col_x}' or '{col_y}' not found.")
            return
        x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        y_num = pd.api.types.is_numeric_dtype(self.df[col_y])
        if x_num and y_num:
            fig = px.scatter(self.df, x=col_x, y=col_y, trendline='ols',
                             title=f'{col_x} vs {col_y}', template='plotly_white')
        elif not x_num and y_num:
            fig = px.box(self.df, x=col_x, y=col_y, points='all',
                         title=f'{col_y} by {col_x}', template='plotly_white')
        elif x_num and not y_num:
            fig = px.box(self.df, x=col_y, y=col_x, points='all',
                         title=f'{col_x} by {col_y}', template='plotly_white')
        else:
            cross = pd.crosstab(self.df[col_x], self.df[col_y])
            fig = px.bar(cross.reset_index().melt(id_vars=col_x),
                         x=col_x, y='value', color=col_y, barmode='group',
                         title=f'{col_x} vs {col_y}', template='plotly_white')
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

    def display_image(self, result):
        self._plotter.display_image(result)

    def plot_bar_chart(self, **kwargs):
        return self._plotter.plot_bar_chart(**kwargs)

    def plot_pie_chart(self, **kwargs):
        return self._plotter.plot_pie_chart(**kwargs)

    def plot_histogram(self, **kwargs):
        return self._plotter.plot_histogram(**kwargs)

    # Statistical Insights

    def plot_numerical_correlation(self):
        if not self._check_df():
            return
        num_df = self.df.select_dtypes(include='number')
        if num_df.shape[1] < 2:
            print("Need at least 2 numeric columns.")
            return
        corr = num_df.corr(method='pearson')
        fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                        zmin=-1, zmax=1, title='Pearson Correlation Heatmap',
                        template='plotly_white')
        fig.update_layout(height=600)
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

    def plot_categorical_correlation(self):
        if not self._check_df():
            return
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        if len(cat_cols) < 2:
            print("Need at least 2 categorical columns.")
            return

        def cramers_v(x, y):
            confusion = pd.crosstab(x, y)
            chi2 = stats.chi2_contingency(confusion)[0]
            n = confusion.sum().sum()
            phi2 = chi2 / n
            r, k = confusion.shape
            phi2c = max(0, phi2 - ((k-1)*(r-1)) / (n-1))
            rc = r - (r-1)**2 / (n-1)
            kc = k - (k-1)**2 / (n-1)
            denom = min(rc-1, kc-1)
            return np.sqrt(phi2c / denom) if denom > 0 else 0.0

        matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
        for c1 in cat_cols:
            for c2 in cat_cols:
                matrix.loc[c1, c2] = cramers_v(self.df[c1].astype(str), self.df[c2].astype(str))
        fig = px.imshow(matrix.astype(float), text_auto='.2f', color_continuous_scale='Blues',
                        zmin=0, zmax=1, title="Cramer's V Heatmap", template='plotly_white')
        fig.update_layout(height=600)
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

    def plot_all_associations_heatmap(self):
        if not self._check_df():
            return
        all_cols = self.df.columns.tolist()
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        n = len(all_cols)
        matrix = np.zeros((n, n))

        def cramers_v(x, y):
            confusion = pd.crosstab(x.astype(str), y.astype(str))
            if confusion.shape[0] < 2 or confusion.shape[1] < 2:
                return 0.0
            chi2 = stats.chi2_contingency(confusion)[0]
            total = confusion.sum().sum()
            phi2 = chi2 / total
            r, k = confusion.shape
            phi2c = max(0, phi2 - ((k-1)*(r-1)) / (total-1))
            rc = r - (r-1)**2 / (total-1)
            kc = k - (k-1)**2 / (total-1)
            denom = min(rc-1, kc-1)
            return np.sqrt(phi2c / denom) if denom > 0 else 0.0

        def eta(numeric_col, cat_col):
            groups = [numeric_col[cat_col == c].dropna() for c in cat_col.unique()
                      if len(numeric_col[cat_col == c].dropna()) > 0]
            if len(groups) < 2:
                return 0.0
            f, _ = stats.f_oneway(*groups)
            k = len(groups)
            n = sum(len(g) for g in groups)
            eta2 = (f * (k-1)) / (f * (k-1) + (n-k))
            return np.sqrt(max(0, eta2))

        for i, c1 in enumerate(all_cols):
            for j, c2 in enumerate(all_cols):
                if i == j:
                    matrix[i, j] = 1.0
                elif c1 in num_cols and c2 in num_cols:
                    valid = self.df[[c1, c2]].dropna()
                    if len(valid) > 1:
                        r, _ = stats.pearsonr(valid[c1], valid[c2])
                        matrix[i, j] = abs(r)
                elif c1 in cat_cols and c2 in cat_cols:
                    matrix[i, j] = cramers_v(self.df[c1], self.df[c2])
                else:
                    num_c = c1 if c1 in num_cols else c2
                    cat_c = c2 if c1 in num_cols else c1
                    valid = self.df[[num_c, cat_c]].dropna()
                    if valid[cat_c].nunique() == 2:
                        cats = valid[cat_c].unique()
                        binary = (valid[cat_c] == cats[0]).astype(int)
                        r, _ = pointbiserialr(binary, valid[num_c])
                        matrix[i, j] = abs(r)
                    else:
                        matrix[i, j] = eta(valid[num_c], valid[cat_c])

        assoc_df = pd.DataFrame(matrix, index=all_cols, columns=all_cols)
        fig = px.imshow(assoc_df, text_auto='.2f', color_continuous_scale='Viridis',
                        zmin=0, zmax=1, title='Unified Association Heatmap',
                        template='plotly_white')
        fig.update_layout(height=700)
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


# Demo using Titanic dataset

def run_demo():
    inspector = DataInspector()
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    inspector.df = pd.read_csv(url)
    inspector._auto_convert_types()

    inspector.get_summary()
    inspector.column_details()
    inspector.show_missing_data()

    inspector.handle_missing_values(strategy='median')
    inspector.remove_duplicates()
    inspector.delete_columns(columns='Cabin,Ticket,Name')
    inspector.handle_outliers(columns=['Fare', 'Age'], find_and_delete=False)

    inspector.plot_numerical(column_names=['Age', 'Fare', 'SibSp'])
    inspector.plot_categorical(column_names=['Sex', 'Pclass', 'Embarked'])
    inspector.plot_relationship('Pclass', 'Fare')
    inspector.plot_relationship('Age', 'Fare')
    inspector.plot_relationship('Sex', 'Embarked')

    final_df = inspector.create_normalized_data_df()
    print("ML-ready DataFrame shape:", final_df.shape)

    inspector.plot_numerical_correlation()
    inspector.plot_categorical_correlation()
    inspector.plot_all_associations_heatmap()

    PLT = PlottingMethods()
    PLT.display_image(PLT.plot_bar_chart(x='Pclass', y='Fare', data=inspector.df, color='Sex'))
    PLT.display_image(PLT.plot_pie_chart(names='Pclass', values='PassengerId', data=inspector.df, hole=0.4))
    PLT.display_image(PLT.plot_histogram(x='Age', data=inspector.df))
    PLT.display_image(PLT.plot_heat_map(values='Fare', index='Sex', columns='Pclass', data=inspector.df))
    PLT.display_image(PLT.plot_sankey_diagram(source_column='Sex', target_column='Pclass', values='Fare', data=inspector.df))
    PLT.display_image(PLT.plot_simple_sunburst_graph(path=['Sex', 'Pclass'], values='Fare', data=inspector.df))

    print("Done!")


if __name__ == '__main__':
    run_demo()

Rows: 891, Columns: 12
Numeric columns (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
Categorical columns (4): ['Name', 'Sex', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,6.610000e+02,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,2.603185e+05,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,4.716093e+05,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,6.930000e+02,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,1.999600e+04,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,2.361710e+05,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,3.477430e+05,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,3.101298e+06,512.329200


,Column,Dtype,Non-Null,Null,Unique
0,PassengerId,int64,891,0,891
1,Survived,int64,891,0,2
2,Pclass,int64,891,0,3
3,Name,object,891,0,891
4,Sex,object,891,0,2
5,Age,float64,714,177,88
6,SibSp,int64,891,0,7
7,Parch,int64,891,0,7
8,Ticket,float64,661,230,514
9,Fare,float64,891,0,248


,Missing Count,Missing %
Cabin,687,77.10
Ticket,230,25.81
Age,177,19.87
Embarked,2,0.22


Missing values handled using 'median'.
Removed 0 duplicate(s). Rows remaining: 891
Deleted: ['Cabin', 'Ticket', 'Name']. Remaining: ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Fare: 116 outliers (range: -26.72 to 65.63)
Age: 66 outliers (range: 2.50 to 54.50)
Total outlier rows: 170. Use find_and_delete=True to remove.


Numeric data scaled using 'minmax'.
Categorical data encoded using 'ordinal'.
Merged DataFrame shape: (891, 9)
ML-ready DataFrame shape: (891, 9)


Done!
